Ce notebook Jupyter sert à récupérer les données Eco2Max depuis Janvier 2021 jusqu'à Décembre 2025.

In [ ]:
# On commence par définir la fonction qui va nous permettre de télécharger les données qui nous intéressent.
from pathlib import Path
import requests

BASE = "https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets"
DATASET = "eco2mix-regional-cons-def"


def download_eco2mix_regional(year: int, raw_dir: Path = Path("data/raw/eco2mix")) -> Path:
    """fonction qui nous permets de télécharger une année d'éCO2mix régional en Parquet."""
    dest = raw_dir / f"eco2mix_regional_{year}.parquet"
    if dest.exists():
        return dest                      # cache : pour éviter qu'on retélécharge
    raw_dir.mkdir(parents=True, exist_ok=True)

    params = {
        "where": f"date >= date'{year}-01-01' AND date <= date'{year}-12-31'",
        "timezone": "UTC",               # tout en UTC pour joindre avec la météo
    } #On va télécharger année après année
    url = f"{BASE}/{DATASET}/exports/parquet"
    tmp = dest.with_suffix(".part")      # évite de garder un fichier à moitié téléchargé
    with requests.get(url, params=params, stream=True, timeout=900) as r:
        r.raise_for_status()
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    tmp.rename(dest)
    return dest